In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from view import View

In [25]:
RSUN = 696e8
A, B, C = 14.712, -2.396, -1.787

def get_velocity(header, **kwargs):
    view = View.from_header(header)

    xi, yi, zi = view.grid(origin='carrington', **kwargs)
    U = (A + B * yi ** 2 + C * yi ** 4) * RSUN / 100 * np.pi / 180 / 24 / 3600

    transform = view.to_heliographic(origin='carrington', **kwargs)
    v, _ = transform((zi * U, 0, -xi * U))
    vx, vy, vz = v[0] - view.vw, v[1] - view.vn, v[2] - view.vr
    xi, yi, zi = view.grid(origin='heliographic', **kwargs)

    q = np.tan(view.rsun_arc * np.pi / 180 / 3600)
    d = np.sqrt(1 - 2 * zi * q + q ** 2)
    V = (q * (xi * vx + yi * vy + zi * vz) - vz) / d
    return V

def get_mu(header, **kwargs):
    view = View.from_header(header)
    xi, yi, zi = view.grid(origin='heliographic', **kwargs)
    return zi

In [129]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/vlos/*20260615*.fits'))

In [130]:
with fits.open(files[0]) as hdul:
    data = hdul[0].data
    header = hdul[0].header

In [131]:
mu = get_mu(header)
velocity = get_velocity(header)

print(np.nanmedian(velocity), header['OBS_VR'], np.nanmedian(velocity - mu * 250) - header['OBS_VR'])

-6249.64815811316 -6250.17644423106 -190.6693630816908


In [136]:
temp = data - velocity + mu * 250 + header['OBS_VR']
temp -= q
print(np.nanmedian(temp))
temp -= np.nanmedian(temp)

plt.figure(figsize=(10,10))
plt.imshow(temp, 'seismic', vmin=-1000, vmax=1000)
plt.tight_layout()

646.0324197915122


In [100]:
q = temp.copy()

In [122]:
plt.figure(figsize=(10,10))
plt.imshow((temp - q) / q_V, 'seismic', vmin=-0.03, vmax=0.03)
plt.tight_layout()

In [108]:
np.nanmedian((temp - q) / q_V)

np.float64(-0.002684610626651346)

In [30]:
plt.figure(figsize=(10,10))
plt.imshow(data, 'seismic', vmin=-3000, vmax=3000)
plt.tight_layout()

In [23]:
q_V = 299792458 / 6173.341

plt.figure(figsize=(10,10))
plt.imshow(data / q_V, 'seismic', vmin=-0.1, vmax=0.1)
plt.tight_layout()

In [94]:
508 / q_V

0.01046076091747445